# Notebook 02 — RAG Pipeline

Demonstrates retrieval-augmented generation against the sample program corpus,
including retrieval failure modes and their mitigations.

<!-- TODO main-session: expand teaching framing; tie back to NB 01 closing arc -->

## Setup

Adds the repo root to `sys.path`, loads environment variables, and imports the public RAG API.

<!-- TODO main-session: expand teaching framing -->

In [1]:
from __future__ import annotations
import os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env", override=False)

from src.rag import ingest, retrieve, RetrievedDocument
from src.llm import LLMClient

provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")

Provider: anthropic · Anthropic key present: True


## Ingest the corpus

Loads the five sample markdown documents, chunks them, embeds them, and persists the vector store.

<!-- TODO main-session: expand teaching framing -->

In [2]:
corpus_dir = repo_root / "data"
persist_dir = repo_root / "data" / "chroma_nb02"

result = ingest(
    corpus_dir=corpus_dir,
    persist_dir=persist_dir,
    chunk_size=500,
    chunk_overlap=50,
)

print(f"documents_loaded  : {result.documents_loaded}")
print(f"chunks_created    : {result.chunks_created}")
print(f"chunks_indexed    : {result.chunks_indexed}")
print(f"vector_store_path : {result.vector_store_path}")
print(f"embedding_model   : {result.embedding_model}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


documents_loaded  : 5
chunks_created    : 42
chunks_indexed    : 42
vector_store_path : c:\Users\narla\OneDrive\Desktop\TalentSprint\IISc_GenAI_C2\LLMOps\llmops-session\data\chroma_nb02
embedding_model   : sentence-transformers/all-MiniLM-L6-v2


## Baseline retrieval

Retrieve the top-5 chunks for a simple policy question and inspect scores + source priority.

<!-- TODO main-session: expand teaching framing -->


In [ ]:
hits = retrieve(persist_dir, "What is the late submission policy?", k=5)

SEP = "-" * 60
for i, doc in enumerate(hits, 1):
    doc_id = doc.chunk.metadata.document_id
    priority = doc.chunk.metadata.source_priority
    print(f"Hit {i}")
    print(f"  document_id     : {doc_id}")
    print(f"  source_priority : {priority}")
    print(f"  score           : {doc.score:.4f}")
    print(f"  text (first 200): {doc.chunk.text[:200]!r}")
    print(SEP)


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


AttributeError: 'DocumentMetadata' object has no attribute 'get'

## Notice the source_priority

The policy doc (`source_priority=1`) ranks at or near the top for a policy question — the retriever naturally surfaces the authoritative source. Later sections show when this breaks down.

<!-- TODO main-session: expand teaching framing -->
